# Sentence embeddings

## 1. Введение

**Эмбеддинг слова** — это вектор, который представляет одно слово. У него есть два больших ограничения:
1. Он не учитывает контекст. Слово «кот» в предложениях «кот спит» и «кот поцарапал диван» получит один и тот же вектор.
2. Из набора векторов слов неочевидно, как получить вектор всего предложения. Простое усреднение — плохая идея, потому что оно не различает смысловые и служебные слова.

**Эмбеддинг предложения** — это один вектор фиксированной длины, который представляет смысл всего предложения целиком. Ключевое свойство: семантически близкие предложения получают близкие векторы, даже если у них нет общих слов.

Например:
- «Кот сидит на коврике»
- «Пушистый зверь устроился на полу»

У этих предложений почти нет общих слов, но их sentence embeddings должны быть близки. Именно это делает sentence embeddings полезными для поиска и сравнения по смыслу, а не по ключевым словам.

**Общая задача Sentence embeddings**: оценить семантическое сходство двух текстов. 

Дана пара текстов (запрос-документ, вопрос-ответ, два предложения), нужно понять, насколько они близки по смыслу.

Есть два принципиально разных способа решить эту задачу:
- **Bi-encoder** — сначала получить вектор для каждого текста, потом сравнить векторы. Это и есть sentence embeddings.
- **Cross-encoder** — подать пару текстов в модель вместе и получить оценку напрямую. Векторов предложений при этом не возникает.

Основная часть этого ноутбука - про первый способ, то есть про то, **как получают sentence embeddings**. Cross-encoder появится ближе к концу как альтернатива и дополнение в контексте reranking.

## 2. Почему BERT не даёт готовых эмбеддингов предложения

BERT — это encoder-модель на основе трансформера. На вход она принимает последовательность токенов, на выходе даёт **последовательность векторов** — по одному на каждый токен:

```text
[CLS]  кот  сидел  на  окне  [SEP]
  ↓     ↓     ↓     ↓    ↓      ↓
 v0    v1    v2    v3   v4     v5
```

Каждый вектор — **контекстуализированный**. Вектор для слова «кот» в предложении «кот сидел на окне» будет отличаться от вектора «кот» в предложении «кот поцарапал диван». Это большое достижение по сравнению со статичными эмбеддингами слов вроде word2vec или GloVe.

Но обрати внимание: **на выходе нет одного вектора для всего предложения**. Информация о предложении распределена по всем токенным векторам.

### Контекстуализированные эмбеддинги слов — что это и чем они полезны

Контекстуализированные эмбеддинги слов хороши для задач, где нужно понять **отдельные слова в контексте**: NER, POS-tagging, извлечение отношений, вопросно-ответные задачи с указанием на фрагмент текста.

Но они **не подходят** для задач, где нужно сравнить два текста целиком и получить одну оценку сходства.

### Попытки получить вектор предложения из BERT

#### `[CLS]`-токен

У BERT есть специальный токен `[CLS]`, который стоит в начале последовательности. При обучении поверх его вектора ставили классификационную голову. Логично было бы использовать `v0` как представление всего предложения.

**Но на практике это работает плохо.** Причины:

- `[CLS]` обучался на задачи классификации, а не на задачи similarity.
- В оригинальном BERT поверх `[CLS]` стояла голова для Next Sentence Prediction, и `[CLS]` приспосабливался под неё.
- Как универсальный sentence embedding `[CLS]` без дообучения даёт почти случайную близость между предложениями.

#### Mean pooling

Второй популярный вариант — усреднить векторы всех токенов:

```text
sentence_vector = mean(v1, v2, ..., vn)
```

Это работает чуть лучше `[CLS]`, но тоже плохо, и вот почему:

- Служебные слова (the, a, is, и, на, в) вносят такой же вклад, как и смысловые.
- Усреднение «размывает» смысл: важные слова теряются среди служебных.
- Главное — **BERT не обучали так, чтобы результат усреднения давал вектор, пригодный для сравнения через косинус**.

### Главная причина

Дело не в том, что BERT «не умеет» кодировать предложение. Информация о предложении в его выходных векторах есть. Проблема в другом:

> **BERT не обучали на то, чтобы близкие по смыслу предложения имели близкие векторы.**

Его предобучали на Masked Language Modeling (угадывание замаскированных слов) и Next Sentence Prediction (следует ли одно предложение за другим). Ни одна из этих задач не требует, чтобы пространство эмбеддингов было выровнено под семантическое сходство.

Именно это и привело к появлению Sentence-BERT и всей парадигмы sentence embeddings: нужна не просто модель, которая понимает контекст, а модель, **специально обученная давать пригодные для сравнения векторы предложений**.

## 3. Что такое sentence embedding

### Один вектор фиксированной длины на текст

Sentence embedding — это вектор фиксированной размерности (например, 384, 512 или 768 чисел), который представляет **смысл всего текста**: предложения, абзаца, иногда целого документа.

Формально:

```text
encode("Кот сидит на коврике")   -> [0.12, -0.87, 0.34, ..., 0.05]   # 384 числа
encode("Пушистый зверь устроился на полу") -> [0.11, -0.85, 0.31, ..., 0.07]
```

Эти два вектора будут близки, потому что предложения близки по смыслу.

### Семантическая близость через косинусное сходство

Чтобы сравнить два вектора, используют косинусное сходство:

```text
similarity(A, B) = cos(A, B) = (A · B) / (|A| * |B|)
```

Значение от -1 до 1. Чем ближе к 1, тем ближе по смыслу. Обычно эмбеддинги нормализуют до единичной длины, и тогда косинус сводится к скалярному произведению.

### Требования к хорошему sentence embedding

1. **Семантика.** Близкие по смыслу тексты должны иметь близкие векторы. Это главное требование, и именно его не выполняет vanilla BERT.
2. **Фиксированная размерность.** Вектор должен быть одной длины независимо от длины текста.
3. **Вычислительная эффективность.** Получение вектора должно быть быстрым, чтобы можно было обрабатывать миллионы текстов.
4. **Возможность предпосчёта.** Вектор текста должен зависеть только от самого текста, а не от того, с чем его сравнивают.

### Что даёт предпосчёт эмбеддингов

Последнее требование — ключевое для практики. Если вектор зависит только от текста, можно:

1. **Один раз посчитать эмбеддинги всех документов** в базе и сохранить их.
2. При появлении запроса посчитать **один** вектор запроса.
3. Быстро найти ближайшие документы через косинусное сходство.

Для миллиона документов это означает: один forward-проход на запрос, а не миллион. Именно это делает sentence embeddings основой современного семантического поиска.

## 4. Bi-encoder: как получают sentence embeddings

### Архитектура

Bi-encoder следует парадигме **Siamese Networks** или **Dual-Encoder**.

Идея: у нас есть **один энкодер**, веса которого используются совместно. Мы прогоняем через него два текста **независимо**:

```text
текст A -> encoder -> pooling -> вектор A
текст B -> encoder -> pooling -> вектор B
score = cosine(вектор A, вектор B)
```

Слово «сиамский» здесь означает, что оба прохода используют одну и ту же модель с одними и теми же весами. Иногда используют два разных энкодера (например, для текста и картинки в мультимодальных задачах) — тогда это **dual encoder** в узком смысле.

Ключевое свойство: **тексты не взаимодействуют между собой внутри модели**. Вектор A не зависит от вектора B. Это и позволяет предпосчитывать эмбеддинги.

### Общие веса для обоих проходов

Общие веса — не деталь реализации, а принципиальное решение. Если бы у нас были две разные модели для запроса и документа, они бы давали векторы в несогласованных пространствах, и косинус между ними не имел бы смысла. Общие веса гарантируют, что оба текста попадают в **одно и то же векторное пространство**.

### Базовая модель

В качестве основы bi-encoder'а берут предобученную модель: BERT, RoBERTa, DistilBERT, MPNet, DeBERTa, LaBSE и другие. Эту модель называют **encoder** или **base model**.

Она даёт контекстуализированные эмбеддинги токенов, а следующие слои превращают их в один вектор предложения.

> [encodechka](https://github.com/avidale/encodechka)

### Слой пулинга

После энкодера у нас есть последовательность векторов токенов. Нужно получить из неё **один вектор** для всего предложения. Этот шаг называется **pooling**.

Основные стратегии:

- **CLS-pooling.** Взять вектор `[CLS]`-токена. Работает, но уступает mean pooling'у.
- **Mean pooling.** Усреднить векторы всех токенов (иногда без `[CLS]`, `[SEP]` и padding-токенов). Популярный и надёжный вариант.
- **Max pooling.** Взять максимум по каждому измерению среди всех токенов. Реже используется, но иногда даёт лучший результат.
- **Mean pooling + нормализация.** Усреднить и привести вектор к единичной длине. **Самая распространённая и эффективная стратегия.** Нормализация важна, потому что после неё косинусное сходство сводится к скалярному произведению, что ускоряет поиск.

```text
mean_pooling:
    v = mean(v1, v2, ..., vn)
    v = v / ||v||
```

Важно понимать: **pooling сам по себе не решает проблему**. Даже лучший pooling над vanilla BERT даст плохие эмбеддинги. Работает только связка **pooling + дообучение на similarity-задачах**.

### Обучение на контрастивных задачах

Самое главное в bi-encoder — как он обучается. Его **не обучают предсказывать слова**. Его обучают понимать **сходство между текстами**.

Идея контрастивного обучения: у нас есть пары или тройки текстов, размеченные по степени сходства. Функция потерь «притягивает» векторы похожих текстов и «отталкивает» векторы непохожих.

#### Contrastive Loss

Подаётся пара предложений:

- **Позитивная пара** (похожие) — loss минимизирует расстояние между векторами.
- **Негативная пара** (непохожие) — loss максимизирует расстояние, но только если оно уже не превышает порог `margin`.

```text
L = y * d(A, B)^2 + (1 - y) * max(0, margin - d(A, B))^2
```

где `y = 1` для позитивной пары и `y = 0` для негативной.

#### Triplet Loss

Подаётся три предложения:

- **Anchor** — якорь.
- **Positive** — похожее на якорь.
- **Negative** — непохожее на якорь.

Модель учится делать так, чтобы расстояние `(Anchor, Positive)` было **меньше**, чем расстояние `(Anchor, Negative)`, на определённую величину `margin`.

```text
L = max(0, d(A, P) - d(A, N) + margin)
```

Triplet loss мощнее contrastive loss, но требует аккуратного подбора триплетов. Случайные негативы часто слишком лёгкие, и модель на них ничему не учится.

#### Multiple Negatives Ranking Loss (MNRL)

Очень популярный и эффективный метод. Работает **внутри батча**.

Идея: для каждого якоря у нас есть одно позитивное предложение, а **все остальные позитивы в батче используются как негативы**. То есть если в батче `N` пар, то для каждой пары есть `1` позитив и `N-1` негативов.

Модель учится ранжировать позитив выше всех негативов. Это даёт **много обучающего сигнала из одного батча** и хорошо работает на больших датасетах.

Этот loss используется по умолчанию в большинстве современных bi-encoder'ов.

### Данные для обучения

Bi-encoder обучают на датасетах с размеченной семантической близостью. Основные типы:

- **Natural Language Inference (NLI).** Пары предложений с пометками:
  - `entailment` — следует, похоже.
  - `contradiction` — противоречит, непохоже.
  - `neutral` — нейтрально.
- **Question-Answer Duplicates.** Пары вопросов, которые считаются дубликатами (например, с StackExchange, Quora).
- **Перефразирования.** Пары предложений, которые являются перефразами друг друга.

На основе этих данных строят позитивные и негативные пары для контрастивных лоссов.

### Sentence Transformers как библиотека

**Sentence Transformers** (`sentence-transformers`) — это Python-библиотека, которая реализует bi-encoder'ы и даёт удобный API для их использования и обучения.

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([
    "Кот сидит на коврике",
    "Пушистый зверь устроился на полу",
])
```

Внутри библиотеки также есть loss-функции (`ContrastiveLoss`, `TripletLoss`, `MultipleNegativesRankingLoss`), тренеры, оценка и интеграция с FAISS для поиска.

## 5. Применения sentence embeddings

Благодаря скорости и возможности предпосчёта, sentence embeddings нашли огромное количество применений:
* Семантический поиск
* Кластеризация текстов
* Поиск дубликатов
* Рекомендательные системы
* RAG (retrieval-этап)
* Перефразирование и оценка текстуального сходства
* Zero-shot классификация

## 6. Ограничения подхода и альтернатива: cross-encoder

### Где sentence embeddings не хватает

Bi-encoder быстр и масштабируем, но у него есть принципиальное ограничение: **тексты не взаимодействуют внутри модели**. Вектор запроса и вектор документа строятся независимо. Это значит, что модель не может «посмотреть» на конкретные слова запроса и документа вместе и уловить тонкие различия.

Например, запрос «не работает звук на ноутбуке» и документ «звук на ноутбуке работает» будут иметь довольно близкие эмбеддинги, хотя смысл противоположный. Bi-encoder часто путается в таких случаях.

Для финального ранжирования, когда кандидатов уже немного, хочется большей точности. Здесь и появляется cross-encoder.

### Cross-encoder: пара текстов подаётся в модель совместно

Cross-encoder подаёт **оба текста в модель одновременно**:

```text
[CLS] запрос [SEP] документ [SEP] -> encoder -> score
```

Модель видит оба текста сразу, и **внимание ходит между всеми токенами пары**. Она может сопоставлять конкретные слова запроса с конкретными словами документа и улавливать тонкие различия.

На выходе — **скаляр**: насколько эта пара релевантна. Не вектор, не эмбеддинг, а одно число.

### Нет предпосчёта

Ключевое отличие: **cross-encoder не создаёт эмбеддинги текстов**. Его выход зависит от всей пары сразу. Нельзя:

- закодировать документ один раз и использовать вектор для разных запросов;
- закэшировать эмбеддинги и искать через FAISS.

Каждую пару `(запрос, документ)` нужно прогонять через модель **отдельно**.

### Дорого и медленно, но точнее

Если в базе `N` документов, то для одного запроса нужно `N` forward-проходов cross-encoder'а. Для миллиона документов это неприемлемо.

Зато точность обычно **заметно выше**, чем у bi-encoder. Cross-encoder лучше различает близкие по форме, но разные по смыслу пары.

### Типичные модели и библиотека

Примеры моделей:

- `cross-encoder/ms-marco-MiniLM-L-6-v2`
- `BAAI/bge-reranker-base`
- `BAAI/bge-reranker-large`

В библиотеке `sentence-transformers` для cross-encoder'ов есть отдельный класс:

```python
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = model.predict([
    ("не работает звук на ноутбуке", "звук на ноутбуке работает"),
    ("не работает звук на ноутбуке", "как починить звук на ноутбуке"),
])
```

### Задача reranking

Cross-encoder идеально подходит для **reranking** — переранжирования уже отобранных кандидатов.

Идея: не прогонять cross-encoder по всей базе, а применить его только к **небольшому списку кандидатов**, которые уже отобрал быстрый метод. Тогда дороговизна cross-encoder'а перестаёт быть проблемой: вместо миллиона пар мы прогоняем сто.

## 7. Как bi-encoder и cross-encoder работают вместе

### Двухэтапная схема: retrieval → reranking

На практике bi-encoder и cross-encoder почти всегда используют **вместе**, в двухэтапной схеме:

1. **Retrieval.** Bi-encoder быстро достаёт **top-N кандидатов** из всей базы. Обычно `N = 50–200`.
2. **Reranking.** Cross-encoder переранжирует этих кандидатов и оставляет **top-K**. Обычно `K = 5–20`.

### Bi-encoder достаёт top-N кандидатов

На этом этапе важна скорость. Bi-encoder кодирует запрос и находит ближайшие документы среди предпосчитанных эмбеддингов. Для миллиона документов это миллисекунды благодаря ANN-поиску (FAISS, HNSW, ScaNN).

Bi-encoder может ошибаться в тонких случаях, но его задача — **не потерять релевантные документы**, а не идеально их отранжировать.

### Cross-encoder переранжирует и оставляет top-K

На этом этапе важна точность. Cross-encoder получает пары `(запрос, документ)` для `N` кандидатов и выдаёт скоры. Сортируем по скору и оставляем `K` лучших.

Поскольку `N` невелико (сотни, а не миллионы), дороговизна cross-encoder'а не критична.

### Баланс скорости и точности

Такая схема даёт лучшее из двух миров:

- Скорость bi-encoder'а на этапе поиска по большой базе.
- Точность cross-encoder'а на этапе финального ранжирования.

Это стандарт в современных поисковых системах и RAG-пайплайнах.

### Примеры из практики

- **Поисковые системы.** BM25 или bi-encoder достаёт кандидатов, cross-encoder переранжирует.
- **RAG-пайплайны.** Bi-encoder ищет релевантные документы, cross-encoder отбирает лучшие, они подаются в LLM.
- **Вопросно-ответные системы.** Bi-encoder ищет похожие вопросы, cross-encoder выбирает наиболее подходящий ответ.
- **Рекомендательные системы.** Bi-encoder отбирает похожие товары, cross-encoder уточняет порядок.

## 8. Тематическое моделирование с BERTopic

### Что такое BERTopic

BERTopic — это библиотека для **тематического моделирования** (topic modeling). Она автоматически определяет, какие темы присутствуют в коллекции документов, и группирует документы по этим темам.

Ключевой момент: **BERTopic не создаёт эмбеддинги сам**. Он берёт готовую модель sentence embeddings и использует её на первом шаге своего пайплайна. Это делает его естественным примером того, как sentence embeddings применяются на практике.

### Как устроен пайплайн BERTopic

BERTopic работает в пять шагов:

1. **Эмбеддинги документов.** BERTopic берёт модель sentence embeddings (по умолчанию — `all-MiniLM-L6-v2`) и кодирует каждый документ в вектор.
2. **Снижение размерности.** UMAP сжимает векторы из 384 (или другой) размерности в 5 измерений.
3. **Кластеризация.** HDBSCAN группирует близкие векторы в кластеры.
4. **Токенизация.** CountVectorizer разбивает документы кластера на слова.
5. **Представление тем.** c-TF-IDF вычисляет, какие слова наиболее характерны для каждого кластера. Эти слова и становятся «названием» темы.

Таким образом, **sentence embeddings — это фундамент BERTopic**. Без качественных эмбеддингов кластеризация будет бессмысленной.

### Почему это важно в контексте sentence embeddings

- BERTopic показывает, **как именно** sentence embeddings применяются для кластеризации на практике — не просто «похожие документы», а «вот какие темы есть в вашей коллекции».
- BERTopic можно использовать с **любой** моделью sentence embeddings. Это подчёркивает модульность подхода: sentence embeddings — универсальный строительный блок, который подходит для разных задач, а не только для поиска.
- Это связывает теорию с реальными инструментами: `sentence-transformers`, `umap-learn`, `hdbscan`, `bertopic`.

### Пример кода

```python
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Можно использовать любую модель sentence embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(documents)

# Посмотреть найденные темы
topic_model.get_topic_info()
```

### Ограничения

- Качество тем напрямую зависит от качества эмбеддингов. Плохая модель — плохие кластеры.
- UMAP и HDBSCAN имеют свои гиперпараметры, которые нужно подбирать под данные.
- Для коротких текстов (твиты, заголовки) результаты часто хуже, чем для длинных.